# Lecture 13 Lab — Deployment & Sim-to-Real
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, matplotlib, PyTorch |
| **Hardware** | CPU is fine — the whole lab runs in about three minutes |
| **Submit** | This completed notebook (see §12) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID

### Important cell below 

You must assign 3 last digits of your student ID to this <code>s_id</code> variable. For example, if your student ID is 6710546988, you must assign
```python
s_id = 988
```
This code will be printed in later cells. **Each output cell that shows incorrect s_id printout will get -1 penalty per cell.**

s_id may be used in some part of code as well, such as setting random seed. So the output for each student may be slightly different. 

In [ ]:
s_id = None  # replace with 3 last digits of your student ID.

## 1. The Last Mile

Twelve weeks of models, and every one of them was judged on a held-out test set drawn from the same pile of data as the training set. That is the right way to develop a model. It is not what happens when you deploy one.

A deployed model meets data produced by a *different physical process*: a different camera, a different lamp, a different lens, a machine that has worn since the data was collected. Nothing in your validation procedure has ever tested that.

Today you will build a surface-defect classifier that is **perfect in simulation and useless on the factory floor**, and then work out what to do about it. Along the way you will find that the model cannot tell you it is broken — and that something far simpler can.

### Learning objectives

- Measure a sim-to-real gap, and show it is not a data-quantity problem.
- Implement domain randomisation and quantify how much of the gap it closes.
- Demonstrate that softmax confidence does **not** detect domain shift.
- Build a label-free input monitor that does.
- Export a model, measure its latency against a real cycle-time budget, and decide whether optimisation is warranted.

## 2. Setup

In [ ]:
import os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(s_id)
torch.manual_seed(s_id)
torch.set_num_threads(2)
warnings.simplefilter("ignore")
print("PyTorch", torch.__version__)

## 3. Part A — Two Domains

### The simulator

Our "simulator" renders a machined surface under perfect, even lighting with a sharp, perfectly square camera. It produces four classes: **ok**, **scratch**, **pit**, **crack**.

This is the data we have. It is free, perfectly labelled, and available in unlimited quantity — which is exactly why people train on it.

In [ ]:
SIZE = 32
CLASSES = ["ok", "scratch", "pit", "crack"]
print("Student ID : "+str(s_id)) # don't chage this code!

def base_image(cls, rng):
    """A clean, perfectly lit render of a machined surface. THE SIMULATOR."""
    img = np.full((SIZE, SIZE), 0.55, dtype=np.float32)
    img += 0.015 * np.sin(np.arange(SIZE)[:, None] * 1.7 + rng.uniform(0, 6))   # tool marks
    img += rng.normal(0, 0.008, (SIZE, SIZE)).astype(np.float32)

    if cls == 1:                                   # scratch: thin bright line
        ang = rng.uniform(0, np.pi)
        cx, cy = rng.uniform(10, 22, 2)
        L = rng.uniform(9, 14)
        for t in np.linspace(-L / 2, L / 2, 60):
            ri, ci = int(round(cy + t * np.sin(ang))), int(round(cx + t * np.cos(ang)))
            if 0 <= ri < SIZE and 0 <= ci < SIZE:
                img[ri, ci] += 0.30
    elif cls == 2:                                 # pit: small dark depression
        cy, cx = rng.uniform(9, 23, 2)
        rad = rng.uniform(2.0, 3.4)
        yy, xx = np.mgrid[0:SIZE, 0:SIZE]
        d = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
        img -= 0.34 * np.exp(-(d ** 2) / (2 * (rad / 1.6) ** 2))
    elif cls == 3:                                 # crack: jagged dark path
        cy, cx = rng.uniform(8, 24), rng.uniform(6, 12)
        for _ in range(int(rng.uniform(13, 19))):
            ri, ci = int(round(cy)), int(round(cx))
            if 0 <= ri < SIZE and 0 <= ci < SIZE:
                img[ri, ci] -= 0.32
                if ri + 1 < SIZE:
                    img[ri + 1, ci] -= 0.14
            cx += rng.uniform(0.8, 1.7)
            cy += rng.uniform(-1.3, 1.3)
    return np.clip(img, 0, 1)


def make_sim(n, seed):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n):
        c = int(rng.integers(4))
        X.append(base_image(c, rng)); y.append(c)
    return np.array(X, dtype=np.float32), np.array(y)


print("simulator ready")

### The factory camera

The real camera differs in five ways. Three of them we could have anticipated; two we did not think of at all. Keep that split in mind — it decides the result of the whole lab.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
def lighting_gradient(img, rng, strength):
    """Uneven illumination - the lamp is off to one side."""
    ang = rng.uniform(0, 2 * np.pi)
    yy, xx = np.mgrid[0:SIZE, 0:SIZE] / SIZE - 0.5
    return img + strength * (np.cos(ang) * xx + np.sin(ang) * yy)


def blur3(img):
    """3x3 box blur - the real lens is slightly out of focus."""
    p = np.pad(img, 1, mode="edge")
    out = np.zeros_like(img)
    for dy in range(3):
        for dx in range(3):
            out += p[dy:dy + SIZE, dx:dx + SIZE]
    return out / 9.0


def rotate_small(img, rng, max_deg):
    """The camera is mounted a few degrees off square."""
    th = np.deg2rad(rng.uniform(-max_deg, max_deg))
    yy, xx = np.mgrid[0:SIZE, 0:SIZE] - (SIZE - 1) / 2
    si = np.clip(np.round(np.cos(th) * yy + np.sin(th) * xx + (SIZE - 1) / 2).astype(int), 0, SIZE - 1)
    sj = np.clip(np.round(-np.sin(th) * yy + np.cos(th) * xx + (SIZE - 1) / 2).astype(int), 0, SIZE - 1)
    return img[si, sj]


def corrupt_real(img, rng):
    """Everything the factory camera does to a perfect render."""
    img = lighting_gradient(img, rng, rng.uniform(0.18, 0.30))  # uneven lamp
    img = 0.80 * img + 0.10                                     # gain + black level
    img = blur3(img)                                            # defocus
    img = rotate_small(img, rng, 7.0)                           # mount angle
    img = img + rng.normal(0, 0.045, (SIZE, SIZE))              # sensor noise
    return np.clip(img, 0, 1).astype(np.float32)


def make_real(n, seed):
    """THE FACTORY CAMERA. Five differences from the simulator."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n):
        c = int(rng.integers(4))
        X.append(corrupt_real(base_image(c, rng), rng)); y.append(c)
    return np.array(X, dtype=np.float32), np.array(y)


# the SAME part, rendered by the simulator and seen by the factory camera
rng_demo = np.random.default_rng(7)
pairs = []
for c in [1, 2, 3, 0, 1, 2]:
    clean = base_image(c, rng_demo)
    pairs.append((clean, corrupt_real(clean, rng_demo), c))

fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for j, (clean, real, c) in enumerate(pairs):
    axes[0, j].imshow(clean, cmap="gray", vmin=0, vmax=1)
    axes[0, j].set_title(CLASSES[c], fontsize=10)
    axes[1, j].imshow(real, cmap="gray", vmin=0, vmax=1)
    for i in range(2):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
axes[0, 0].set_ylabel("SIMULATOR", fontsize=11)
axes[1, 0].set_ylabel("FACTORY", fontsize=11)
plt.tight_layout(); plt.show()
print("Each column is the SAME part: top as the simulator renders it, bottom as the camera sees it.")
print("You can still read every defect. The question is whether a CNN trained only on row 1 can.")

## 4. Part B — Train in Simulation, Deploy to the Factory

A small CNN, the kind you would actually put on an inspection station.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
class DefectCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.1), nn.Linear(32, 4))

    def forward(self, x):
        return self.net(x)


def train(X, y, epochs=10, seed=0, bs=64):
    torch.manual_seed(seed)
    model = DefectCNN()
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    lossf = nn.CrossEntropyLoss()
    Xt, yt = torch.tensor(X).unsqueeze(1), torch.tensor(y)
    g = torch.Generator().manual_seed(seed)
    model.train()
    for _ in range(epochs):
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            lossf(model(Xt[idx]), yt[idx]).backward()
            opt.step()
    model.eval()
    return model


@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return float((model(torch.tensor(X).unsqueeze(1)).argmax(1).numpy() == y).mean())


@torch.no_grad()
def prediction_counts(model, X):
    model.eval()
    p = model(torch.tensor(X).unsqueeze(1)).argmax(1).numpy()
    return np.bincount(p, minlength=4)


X_sim_test, y_sim_test = make_sim(600, 2)
X_real_test, y_real_test = make_real(600, 3)

t0 = time.time()
model_clean = train(*make_sim(2500, 10))
print(f"trained in {time.time()-t0:.0f}s\n")
print(f"accuracy on SIMULATED test data : {accuracy(model_clean, X_sim_test, y_sim_test):.3f}")
print(f"accuracy on REAL      test data : {accuracy(model_clean, X_real_test, y_real_test):.3f}")
print(f"chance = 0.250")

> Ship that and you would have reported a perfect validation score to your supervisor.
>
> Accuracy alone does not tell you *how* it fails, and that matters for diagnosis. Look at which classes it actually predicts.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
print("true class balance :", np.bincount(y_real_test, minlength=4), " ", CLASSES)
print("model's predictions:", prediction_counts(model_clean, X_real_test))

> The model has essentially collapsed onto one or two classes. The lighting gradient and blur have shifted every image far enough that one class's features dominate everywhere.
>
> This matters practically: **a collapsed prediction distribution is visible without a single label.** Remember that — it comes back in §8.

## 5. Part C — The Obvious Fix That Does Not Work

The instinct is to collect more data. Simulated data is free, so take four times as much.

Think about what you expect before running this.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
model_more = train(*make_sim(10000, 20))
print(f"{'training set':>22}{'sim':>8}{'REAL':>8}")
print(f"{'clean sim  2,500':>22}{accuracy(model_clean, X_sim_test, y_sim_test):>8.3f}"
      f"{accuracy(model_clean, X_real_test, y_real_test):>8.3f}")
print(f"{'clean sim 10,000':>22}{accuracy(model_more, X_sim_test, y_sim_test):>8.3f}"
      f"{accuracy(model_more, X_real_test, y_real_test):>8.3f}")

> Four times the data buys nothing, because the problem was never a shortage of examples. Every one of those 10,000 images came from the same simulator, and the simulator is what is wrong.
>
> **More data from the wrong distribution is still the wrong distribution.** This is the single most expensive mistake in applied ML: months of data collection that could not possibly have helped.

## 6. Part D — Domain Randomisation

If you cannot make the simulator match reality, make it *refuse to commit* to any one appearance. Randomise the things you are unsure about during training, and the model is forced to rely on the defect's shape rather than on the lighting.

Below, `randomise_narrow` is written for you: it varies lighting, gain and noise.

**Your task:** write `randomise_wide`, which varies those *and* the two things we failed to anticipate — defocus blur and mount angle. Use `blur3` and `rotate_small` from §3.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
def randomise_narrow(n, seed):
    """Randomise lighting, gain and noise - the differences we anticipated."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n):
        c = int(rng.integers(4))
        img = base_image(c, rng)
        img = lighting_gradient(img, rng, rng.uniform(-0.35, 0.35))
        img = rng.uniform(0.70, 1.15) * img + rng.uniform(-0.05, 0.15)
        img = img + rng.normal(0, rng.uniform(0.0, 0.07), (SIZE, SIZE))
        X.append(np.clip(img, 0, 1).astype(np.float32)); y.append(c)
    return np.array(X, dtype=np.float32), np.array(y)


def randomise_wide(n, seed):
    """TODO: everything randomise_narrow does, PLUS blur and rotation."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n):
        c = int(rng.integers(4))
        img = base_image(c, rng)
        img = lighting_gradient(img, rng, rng.uniform(-0.35, 0.35))
        img = rng.uniform(0.70, 1.15) * img + rng.uniform(-0.05, 0.15)

        # TODO (2 lines):
        #   1. with probability 0.5, apply blur3(img)        - sometimes out of focus
        #   2. apply rotate_small(img, rng, 10.0)            - random mount angle
        #   Note 10.0 > the real camera's 7 degrees: randomise WIDER than you think you need.

        img = img + rng.normal(0, rng.uniform(0.0, 0.07), (SIZE, SIZE))
        X.append(np.clip(img, 0, 1).astype(np.float32)); y.append(c)
    return np.array(X, dtype=np.float32), np.array(y)


# fail fast if the TODO is not done: wide randomisation must change image statistics
_a, _ = randomise_narrow(200, 0)
_b, _ = randomise_wide(200, 0)
assert abs(_a.std(axis=(1, 2)).mean() - _b.std(axis=(1, 2)).mean()) > 1e-3, \
    "randomise_wide looks identical to randomise_narrow - the TODO is not filled in"
print("randomise_wide implemented")

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
model_narrow = train(*randomise_narrow(2500, 30))
model_wide   = train(*randomise_wide(2500, 40))

print(f"{'training set':>24}{'sim':>8}{'REAL':>8}   predictions on real")
for name, m in [("clean sim", model_clean), ("clean sim x4", model_more),
                ("narrow randomisation", model_narrow), ("wide randomisation", model_wide)]:
    print(f"{name:>24}{accuracy(m, X_sim_test, y_sim_test):>8.3f}"
          f"{accuracy(m, X_real_test, y_real_test):>8.3f}   {prediction_counts(m, X_real_test)}")
print(f"\n{'true balance':>24}{'':>16}   {np.bincount(y_real_test, minlength=4)}")

> **The gap closes in proportion to what you randomised over.** Narrow randomisation fixes the part of the shift it covers and leaves the rest; wide randomisation, which includes blur and rotation, recovers most of the loss and restores a sensible prediction distribution.
>
> Be careful about the lesson here. We knew the answer: we built the factory camera ourselves, so we knew exactly which corruptions to add. **In a real project you do not get that.** You randomise more aggressively than seems reasonable, you accept that some of the gap will remain, and you validate on real data before trusting any of it.
>
> Note too that wide randomisation does not reach the simulated score. Forcing a model to tolerate blur and rotation costs accuracy on clean images. That trade is almost always worth taking.

## 7. Part E — Does the Model Know It Is in Trouble?

Here is the question that decides whether any of this is safe.

A deployed classifier outputs a probability. If that probability dropped when the model went out of its depth, deployment would be easy: gate on confidence, and route low-confidence parts to a human.

Test it. Compute the maximum softmax probability — the model's own stated confidence — on both domains, using the model trained on clean simulation.

In [ ]:
@torch.no_grad()
def confidence(model, X):
    """TODO: return the model's confidence in its chosen class, one value per image.

    1. logits  = model(torch.tensor(X).unsqueeze(1))
    2. probs   = torch.softmax(logits, dim=1)
    3. return the MAXIMUM probability per row, as a numpy array
       (probs.max(1).values.numpy())
    """
    model.eval()
    return None


conf_sim  = confidence(model_clean, X_sim_test)
conf_real = confidence(model_clean, X_real_test)
assert conf_sim is not None, "confidence() still returns None - fill in the TODO"
assert conf_sim.shape == (len(X_sim_test),), "expected one confidence value per image"
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"{'':>12}{'accuracy':>10}{'mean confidence':>18}{'frac over 0.90':>16}")
for name, X, y, c in [("simulated", X_sim_test, y_sim_test, conf_sim),
                      ("REAL",      X_real_test, y_real_test, conf_real)]:
    print(f"{name:>12}{accuracy(model_clean, X, y):>10.3f}{c.mean():>18.3f}{(c > 0.9).mean():>16.3f}")

In [ ]:
with torch.no_grad():
    pred_real = model_clean(torch.tensor(X_real_test).unsqueeze(1)).argmax(1).numpy()
wrong = pred_real != y_real_test
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"on REAL data the model is wrong {wrong.mean():.1%} of the time,")
print(f"and when it is wrong its mean confidence is {conf_real[wrong].mean():.3f}")

> **This is the most important result in the lab.** The model performs at chance on the factory floor and reports high confidence while doing so. It is confidently, catastrophically wrong.
>
> The reason is structural, not a quirk of this network. Softmax is a *normalised* quantity: it reports which class fits best relative to the other three, never whether the input resembles anything in training. Push an image far outside the training distribution and one class still wins — often by a wide margin.
>
> So a confidence threshold is not a safety mechanism against domain shift. Any deployment design that leans on one is resting on nothing.

## 8. Part F — A Monitor That Actually Works

The model cannot see the problem because it only ever looks at images through features tuned for classification. But the shift was obvious in the raw pixels — recall the collapsed prediction counts in §4, and look again at the two rows of images in §3.

So monitor the **input**, not the output. Build the monitor from training data alone, then flag anything that falls outside the range the model was trained on. No labels, no ground truth, works from day one.

In [ ]:
X_train_ref, _ = make_sim(2500, 10)          # exactly what model_clean was trained on
train_contrast = X_train_ref.std(axis=(1, 2))
LO, HI = np.percentile(train_contrast, [0.5, 99.5])
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"per-image contrast in training: mean {train_contrast.mean():.4f}")
print(f"accept band (0.5th-99.5th pct): [{LO:.4f}, {HI:.4f}]")


def flag_rate(X):
    """TODO: fraction of images whose per-image contrast falls OUTSIDE [LO, HI].

    1. contrast = X.std(axis=(1, 2))          # one value per image
    2. outside  = (contrast < LO) | (contrast > HI)
    3. return outside.mean()
    """
    return None


r_sim, r_real = flag_rate(X_sim_test), flag_rate(X_real_test)
assert r_sim is not None, "flag_rate() still returns None - fill in the TODO"
print(f"\nflagged as out-of-distribution:")
print(f"   simulated data : {r_sim:.1%}   (these are in-domain - should be near zero)")
print(f"   REAL data      : {r_real:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(train_contrast, bins=40, alpha=.75, label="training (simulated)", color="#3E5C76")
ax.hist(X_real_test.std(axis=(1, 2)), bins=40, alpha=.75, label="deployed (real)", color="#FF6A39")
ax.axvline(LO, ls="--", c="#1B2A41"); ax.axvline(HI, ls="--", c="#1B2A41")
ax.set_xlabel("per-image contrast (std of pixel values)"); ax.set_ylabel("count")
ax.set_title("The shift the model could not see, visible in one scalar")
ax.legend(); plt.tight_layout(); plt.show()

> One number per image, computed in microseconds, and the two domains barely overlap. The monitor catches almost every real image while leaving in-domain data alone.
>
> Compare the two approaches honestly:
>
> | | detects the shift? | needs labels? | cost |
> |---|---|---|---|
> | softmax confidence | **no** — 92% confident while wrong | no | one forward pass |
> | input statistics | **yes** — flags nearly all of it | no | microseconds |
>
> Contrast is the right statistic *here* because we know the shift involves lighting and noise. A production monitor would track several: per-image mean and contrast, the prediction distribution from §4, and the rate of the rarest class. The principle is general — **watch the data, not the model's opinion of itself.**

## 9. Part G — Shipping It

Now the part that looks like engineering: get the model out of Python and onto the line.

### Export

`torch.export` captures the model as a standalone program. One trap, worth knowing before it costs you an afternoon: export with a batch of 1 and PyTorch bakes in a batch size of 1, and the exported program will refuse every other batch size at runtime. Export with a representative batch and declare the batch dimension dynamic.

In [ ]:
model_wide.eval()
example = (torch.randn(2, 1, SIZE, SIZE),)                     # batch 2, not 1
batch_dim = torch.export.Dim("batch")
exported = torch.export.export(model_wide, example, dynamic_shapes={"x": {0: batch_dim}})
torch.export.save(exported, "/tmp/defect_model.pt2")
torch.save(model_wide.state_dict(), "/tmp/defect_model.pt")

reloaded = torch.export.load("/tmp/defect_model.pt2").module()
with torch.no_grad():
    probe = torch.tensor(X_sim_test[:64]).unsqueeze(1)
    delta = (model_wide(probe) - reloaded(probe)).abs().max().item()
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"parameters            {sum(p.numel() for p in model_wide.parameters()):8d}")
print(f"state_dict            {os.path.getsize('/tmp/defect_model.pt')/1024:8.1f} KB")
print(f"exported program      {os.path.getsize('/tmp/defect_model.pt2')/1024:8.1f} KB")
print(f"max |eager - exported| on a batch of 64: {delta:.2e}")

### Latency against a real budget

A number like "0.3 ms" means nothing until you compare it with the cycle time of the machine it feeds. Our inspection station sees **120 parts per minute** — a 500 ms budget per part — and the camera is 1280 × 1024, so a full part is 1,280 patches of 32 × 32.

In [ ]:
def benchmark(m, x, n=200):
    with torch.no_grad():
        for _ in range(30):
            m(x)
        t0 = time.perf_counter()
        for _ in range(n):
            m(x)
    return (time.perf_counter() - t0) / n * 1000


ms_1  = benchmark(model_wide, torch.randn(1, 1, SIZE, SIZE))
ms_32 = benchmark(model_wide, torch.randn(32, 1, SIZE, SIZE), 60) / 32
PATCHES, BUDGET_MS = (1280 // 32) * (1024 // 32), 500.0
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"{'':>10}{'ms/patch':>11}{'ms/part':>10}{'budget used':>14}")
for name, ms in [("batch=1", ms_1), ("batch=32", ms_32)]:
    per_part = ms * PATCHES
    print(f"{name:>10}{ms:>11.3f}{per_part:>10.0f}{per_part / BUDGET_MS:>13.0%}")
print(f"\nbatching speedup: {ms_1 / ms_32:.1f}x")

### Is quantisation worth it?

Quantisation stores weights as 8-bit integers instead of 32-bit floats. The folklore is that it makes models 4× smaller and faster. Measure it rather than believing it.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
q_model = torch.quantization.quantize_dynamic(model_wide, {nn.Linear}, dtype=torch.qint8)
torch.save(q_model.state_dict(), "/tmp/defect_q.pt")
print("our CNN:")
print(f"   fp32 {os.path.getsize('/tmp/defect_model.pt')/1024:7.1f} KB"
      f"   ->   int8 {os.path.getsize('/tmp/defect_q.pt')/1024:7.1f} KB")

big = nn.Sequential(nn.Flatten(), nn.Linear(SIZE * SIZE, 512), nn.ReLU(),
                    nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 4))
torch.save(big.state_dict(), "/tmp/big.pt")
big_q = torch.quantization.quantize_dynamic(big, {nn.Linear}, dtype=torch.qint8)
torch.save(big_q.state_dict(), "/tmp/big_q.pt")
xb = torch.randn(1, 1, SIZE, SIZE)
print("\na 790k-parameter MLP:")
print(f"   fp32 {os.path.getsize('/tmp/big.pt')/1024:7.0f} KB"
      f"   ->   int8 {os.path.getsize('/tmp/big_q.pt')/1024:7.0f} KB"
      f"   ({os.path.getsize('/tmp/big.pt')/os.path.getsize('/tmp/big_q.pt'):.1f}x smaller)")
print(f"   latency  fp32 {benchmark(big, xb):.3f} ms   ->   int8 {benchmark(big_q, xb):.3f} ms")

> Two honest findings.
>
> On our CNN quantisation achieves **nothing**: dynamic quantisation only converts `Linear` layers, and this network's only `Linear` is a 32 × 4 head. Almost all the weights are in convolutions it never touches.
>
> On a large MLP it does shrink the file about fourfold — and on this CPU it runs no faster, sometimes slower, because int8 kernels are not always the quicker path. **Quantisation buys memory. It does not automatically buy speed.**
>
> Now look back at the latency table, because it contains the real lesson. A single patch takes well under a millisecond, which sounds like plenty of margin — but a full part is 1,280 patches, and at batch 1 that consumes most of the 500 ms cycle time with nothing left for the camera, the conveyor or a bad day. The system genuinely needed optimising.
>
> It just did not need *quantisation*. Batching alone bought a threefold speedup, for free, and brought utilisation down to a comfortable third of the budget. Quantisation would have delivered nothing on this architecture, and on a larger one it ran slower.
>
> So an engineer could have spent the sprint on int8 kernels, achieved nothing on either axis, and shipped a model that was still performing at chance while reporting 92% confidence. **Measure before you optimise, pick the optimisation that the measurement points to, and make sure the thing you are optimising is the thing that is actually broken.**

## 10. Part H — What This Looks Like on the Robot

You will not ship a notebook. In a ROS 2 system the model becomes a node that subscribes to a camera topic and publishes an inspection verdict. The shape of it, with this week's lessons built in:

```python
class DefectInspector(Node):
    def __init__(self):
        super().__init__("defect_inspector")
        self.model = torch.export.load("defect_model.pt2").module()
        self.lo, self.hi = LO, HI                      # monitor band from TRAINING data
        self.sub = self.create_subscription(Image, "/camera/image_raw", self.on_image, 10)
        self.pub = self.create_publisher(String, "/inspection/verdict", 10)

    def on_image(self, msg):
        patches = to_patches(msg)                      # (N, 1, 32, 32) float32

        # 1. GATE FIRST - never classify an image you do not recognise
        contrast = patches.std(axis=(2, 3)).ravel()
        if np.mean((contrast < self.lo) | (contrast > self.hi)) > 0.10:
            self.pub.publish(String(data="REJECT: input out of distribution"))
            return

        # 2. batch the whole part - 3x cheaper than one patch at a time
        with torch.no_grad():
            pred = self.model(torch.from_numpy(patches)).argmax(1)

        self.pub.publish(String(data=summarise(pred)))
```

Three things carried over from the lab: the monitor runs **before** the model and can veto it; patches are batched, because that was the optimisation that mattered; and the exported program is loaded directly, with no model class definition needed on the robot.

What this sketch does not show, and you should insist on in a real system: log every input statistic and the prediction distribution so §4's collapse would be visible on a dashboard; keep a conventional fallback (a fixed-threshold vision check) for when the gate fires; and make sure a human sees the rejects.

## 11. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. Training on four times as much simulated data did not help at all. Explain why, and state what the result tells you about the relationship between dataset size and distribution.**

*Your answer:*

**2. Wide randomisation beat narrow randomisation on real data. Explain the mechanism. Then explain why you could not have relied on this in a real project.**

*Your answer:*

**3. The model was about 92% confident while performing at chance. Explain why softmax confidence fails to detect domain shift, referring to what the softmax actually normalises over.**

*Your answer:*

**4. The input monitor needed no labels and almost no computation, yet caught what the model could not. Propose one additional statistic you would monitor in production, and say what failure it would catch that contrast would miss.**

*Your answer:*

**5. Quantisation made a large model 4× smaller but no faster. Given the latency table you measured, would you have quantised this model? Justify your answer with the numbers.**

*Your answer:*

**6. You are handed this inspection system and told it goes live on Monday on a line where a missed crack reaches a customer. List three things you would require before agreeing, and for each say which result from this lab motivates it.**

*Your answer:*

## 12. Deliverables & Submission

- This notebook, completed and running top-to-bottom without errors (`Kernel → Restart & Run All`).
- The sim-vs-real image grid (§3), the four-way comparison table (§6), the confidence result (§7), the monitor histogram (§8), and the latency table (§9).
- Written answers to the six reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 13. Grading Rubric Guide

| Component | Weight |
|---|---|
| `randomise_wide` correct; four-way comparison produced | 25% |
| `confidence` correct; the confidently-wrong result demonstrated | 25% |
| `flag_rate` correct; monitor histogram produced and interpreted | 20% |
| Export and latency measured; quantisation decision justified with numbers | 15% |
| Reflection questions | 10% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 5% |

**Note :** Actual weights are subjected to Grader's discretion.

---
This is the last lab for this course.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>